In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path
# Change to project root directory (parent of scripts folder)
current_dir = Path.cwd()
if current_dir.name == 'scripts':
    os.chdir(current_dir.parent)


In [ ]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock

# ====== 配置 ======
BASE_URL = "https://www.marxists.org/chinese/maozedong/"
INDEX_URL = "https://www.marxists.org/chinese/maozedong/index.htm"
SAVE_DIR = "data_mzd/documents/"
HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

os.makedirs(SAVE_DIR, exist_ok=True)

def get_soup(url):
    r = requests.get(url, headers=HEADERS, timeout=15)
    r.encoding = "gb2312"   # 非常关键！
    return BeautifulSoup(r.text, "html.parser")

def clean_text(text):
    lines = [line.strip() for line in text.splitlines()]
    return "\n".join(line for line in lines if line)

def extract_article(url):
    soup = get_soup(url)

    # Extract title: try p.title1 first, then h1/h2, then title tag
    title = None
    title_p = soup.find("p", class_="title1")
    if title_p:
        title = title_p.get_text(strip=True)
    else:
        for tag in ["h1", "h2"]:
            t = soup.find(tag)
            if t:
                title = t.get_text(strip=True)
                break
        if not title:
            title = soup.title.get_text(strip=True) if soup.title else ""

    # Extract content from body, excluding navigation and footnotes
    content = ""
    body = soup.find("body")
    if body:
        # Create a copy to avoid modifying the original
        body_copy = BeautifulSoup(str(body), "html.parser").find("body")
        
        # Remove navigation links (usually at the beginning)
        for a in body_copy.find_all("a", href=True):
            # Remove links that look like navigation (containing index, etc.)
            if "index" in a.get("href", "").lower():
                a.decompose()
        
        # Remove the first A tag (usually navigation)
        first_a = body_copy.find("a")
        if first_a:
            first_a.decompose()
        
        # Remove hr and everything after it (usually footnotes/annotations)
        hr = body_copy.find("hr")
        if hr:
            # Remove hr and all siblings after it
            for sibling in hr.find_next_siblings():
                sibling.decompose()
            hr.decompose()
        
        # Remove blockquote (editor's note) if present
        blockquote = body_copy.find("blockquote")
        if blockquote:
            blockquote.decompose()
        
        # Remove title and date paragraphs (already extracted)
        for p in body_copy.find_all("p", class_=["title1", "date"]):
            p.decompose()
        
        # Get text content
        content = body_copy.get_text("\n", strip=True)
    
    # Fallback: try pre tag or div.border if body extraction failed
    if not content:
        pre = soup.find("pre")
        if pre:
            content = pre.get_text("\n", strip=True)
        else:
            div = soup.find("div", class_="border")
            if div:
                content = div.get_text("\n", strip=True)

    content = clean_text(content)
    return title, content

# Thread-safe counter for tracking progress
counter = 0
counter_lock = Lock()

def download_single_article(url, index):
    """Download a single article and save it to disk"""
    global counter
    try:
        title, content = extract_article(url)
        if not content:
            with counter_lock:
                counter += 1
            print(f"[跳过 {counter}] 无正文：{url}")
            return None

        safe_title = title.replace("/", "_").replace("\\", "_")
        filename = f"{index:03d}_{safe_title}.txt"
        path = os.path.join(SAVE_DIR, filename)

        with open(path, "w", encoding="utf-8") as f:
            f.write(title + "\n\n")
            f.write(content)

        with counter_lock:
            counter += 1
        print(f"[{counter}] 已保存：{title}")
        return title

    except Exception as e:
        with counter_lock:
            counter += 1
        print(f"[错误 {counter}] {url} -> {e}")
        return None

def download_mao(max_workers=10):
    """Download articles in parallel"""
    global counter
    counter = 0
    
    index_soup = get_soup(INDEX_URL)

    links = set()
    for a in index_soup.find_all("a", href=True):
        href = a["href"]
        if href.endswith(".htm") and "index" not in href:
            full_url = urljoin(BASE_URL, href)
            links.add(full_url)

    sorted_links = sorted(links)
    print(f"共发现文章链接：{len(sorted_links)}")
    print(f"使用 {max_workers} 个线程并行下载...")

    # Create list of (url, index) tuples for parallel processing
    tasks = [(url, i + 1) for i, url in enumerate(sorted_links)]

    # Download articles in parallel
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(download_single_article, url, idx): (url, idx) 
                   for url, idx in tasks}
        
        # Wait for all tasks to complete
        for future in as_completed(futures):
            future.result()  # This will raise any exceptions that occurred

    print(f"\n下载完成！共处理 {counter} 篇文章")


download_mao(max_workers=10)

In [ ]:
# Split documents into smaller chunks for processing and indexing
from lib.data.chunker import NaiveChunker

chunker = NaiveChunker(
    input_dir="./data_mzd/documents/",
    output_dir="./data_mzd/chunks/",
    chunk_size=500,
    chunk_overlap=100,
    reload=False
)
chunker.run()

In [ ]:
# Extract titles from documents for title-based search indexing
from lib.data.title_extractor import TitleExtractor

extractor = TitleExtractor(
    input_dir="./data_mzd/documents/",
    output_file="./data_mzd/titles.json"
)
extractor.run()

In [ ]:
# LLM Generate summaries for documents using LLM to create concise representations
# cost $1
from lib.data.summary_extractor import SummaryExtractor
import dotenv
dotenv.load_dotenv()

extractor = SummaryExtractor(
    input_dir="./data_mzd/documents/",
    output_file="./data_mzd/summaries.json",
    max_workers=8,
    limit=None
)
await extractor.run(skip_existing=True)

In [ ]:
# Initialize Elasticsearch client for indexing and searching document chunks
from lib.search.elastic_chunk_index import ElasticWriteClientChunks
elastic_chunk = ElasticWriteClientChunks(
    chunk_index_name="mzd",
    chunks_path="./data_mzd/chunks/",
    contexts_path="./data_mzd/contexts/",
    title_path="./data_mzd/titles.json",
    summaries_path="./data_mzd/summaries.json"
)
# Clear existing chunk index and rebuild it with all document chunks
elastic_chunk.clear_index()
elastic_chunk.insert_chunks(
    batch_size=1000,
    limit=None,
    skip_existing=True
)

In [ ]:
# Initialize Elasticsearch client for indexing and searching document titles
from lib.search.elastic_title_index import ElasticWriteClientTitles
elastic_title = ElasticWriteClientTitles(
    title_index_name="mzd_titles",
    title_path="./data_mzd/titles.json",
    summaries_path="./data_mzd/summaries.json"
)
elastic_title.clear_index()
elastic_title.insert_titles(
    limit=None,
    skip_existing=True
)